In [ ]:
# =====================================================
# One-Class SVM for EEG Eye State
# STRICT temporal + ANY-based labeling
# Metrics: ROC-AUC, PR-AUC, Recall@FAR,
#          FAR@Recall, Event Detection Rate, False Alarms / Minute
# =====================================================

# -----------------------------
# 1. FIX RANDOMNESS
# -----------------------------
import numpy as np
import random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# -----------------------------
# 2. IMPORTS
# -----------------------------
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, average_precision_score

# -----------------------------
# 3. LOAD DATA (UPLOAD)
# -----------------------------
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
data = pd.read_csv(file_name)

X_raw = data.iloc[:, :-1].values
y_raw = data.iloc[:, -1].values

# -----------------------------
# 4. SCALE BEFORE WINDOWING
# -----------------------------
scaler = StandardScaler()
X_raw = scaler.fit_transform(X_raw)

# -----------------------------
# 5. STRICT TEMPORAL SPLIT
# -----------------------------
split_idx = int(0.8 * len(X_raw))

X_train_raw = X_raw[:split_idx]
y_train_raw = y_raw[:split_idx]

X_test_raw  = X_raw[split_idx:]
y_test_raw  = y_raw[split_idx:]

# -----------------------------
# 6. WINDOWING (ANY-based)
# -----------------------------
WINDOW_SIZE = 256
TRAIN_STRIDE = 16
TEST_STRIDE  = 256

def create_windows_any(X, y, window, stride):
    Xw, yw = [], []
    for i in range(0, len(X) - window + 1, stride):
        Xw.append(X[i:i + window])
        yw.append(int(np.any(y[i:i + window])))
    return np.array(Xw), np.array(yw)

X_train_win, y_train_win = create_windows_any(
    X_train_raw, y_train_raw, WINDOW_SIZE, TRAIN_STRIDE
)

X_test_win, y_test_win = create_windows_any(
    X_test_raw, y_test_raw, WINDOW_SIZE, TEST_STRIDE
)

# -----------------------------
# 7. TRAIN ONLY ON NORMAL WINDOWS
# -----------------------------
X_train_norm = X_train_win[y_train_win == 0]

X_train_flat = X_train_norm.reshape(len(X_train_norm), -1)
X_test_flat  = X_test_win.reshape(len(X_test_win), -1)

# -----------------------------
# 8. ONE-CLASS SVM
# -----------------------------
ocsvm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

ocsvm.fit(X_train_flat)

# -----------------------------
# 9. ANOMALY SCORES
# -----------------------------
scores = -ocsvm.decision_function(X_test_flat)  # higher = more anomalous

# -----------------------------
# 10. PRIMARY METRICS
# -----------------------------
roc_auc = roc_auc_score(y_test_win, scores)
pr_auc  = average_precision_score(y_test_win, scores)

# -----------------------------
# 11. RECALL @ FAR
# -----------------------------
def recall_at_far(y_true, scores, far):
    normal_scores = scores[y_true == 0]
    threshold = np.percentile(normal_scores, 100 * (1 - far))
    y_pred = (scores >= threshold).astype(int)

    if np.sum(y_true == 1) == 0:
        return np.nan

    recall = np.sum((y_pred == 1) & (y_true == 1)) / np.sum(y_true == 1)
    return recall

recall_far_1 = recall_at_far(y_test_win, scores, far=0.01)
recall_far_5 = recall_at_far(y_test_win, scores, far=0.05)

# -----------------------------
# 12. FAR @ FIXED RECALL
# -----------------------------
def far_at_recall(y_true, scores, target_recall=0.8):
    thresholds = np.sort(scores)
    fars = []

    for th in thresholds:
        y_pred = (scores >= th).astype(int)

        tp = np.sum((y_pred == 1) & (y_true == 1))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        tn = np.sum((y_pred == 0) & (y_true == 0))

        if (tp + fn) == 0:
            continue

        recall = tp / (tp + fn)
        far = fp / (fp + tn)

        if recall >= target_recall:
            fars.append(far)

    return np.min(fars) if len(fars) > 0 else np.nan

far_at_80 = far_at_recall(y_test_win, scores, target_recall=0.8)

# -----------------------------
# 13. EVENT DETECTION RATE (EDR)
# -----------------------------
# An event is detected if ANY anomalous window overlaps it
event_detected = np.any((scores > np.percentile(scores[y_test_win == 0], 95)) & (y_test_win == 1))
edr = 1.0 if event_detected else 0.0

# -----------------------------
# 14. FALSE ALARMS PER MINUTE
# -----------------------------
false_alarms = np.sum((scores > np.percentile(scores[y_test_win == 0], 95)) & (y_test_win == 0))

# assuming 256 samples ≈ 1 second (adjust if needed)
seconds = len(y_test_win)
false_alarms_per_min = (false_alarms / seconds) * 60

# -----------------------------
# 15. FINAL RESULTS
# -----------------------------
print("\n===== ONE-CLASS SVM (FINAL METRICS) =====")
print(f"ROC-AUC              : {roc_auc:.4f}")
print(f"PR-AUC               : {pr_auc:.4f}")
print(f"Recall @ FAR = 1%    : {recall_far_1:.4f}")
print(f"Recall @ FAR = 5%    : {recall_far_5:.4f}")
print(f"FAR @ Recall = 80%   : {far_at_80:.4f}")
print(f"Event Detection Rate : {edr:.2f}")
print(f"False Alarms / Minute: {false_alarms_per_min:.2f}")
# ===============================
# ROC CURVE — One-Class SVM
# ===============================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from matplotlib.font_manager import FontProperties

# Compute ROC curve
fpr, tpr, _ = roc_curve(y_test_win, scores)

plt.figure(figsize=(8, 6), dpi=600)

plt.plot(
    fpr, tpr,
    linewidth=3,
    label=f"One-Class SVM (AUC = {roc_auc:.3f})"
)

plt.plot([0, 1], [0, 1], 'k--', linewidth=4)

# Labels & title
plt.xlabel("False Positive Rate", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("True Positive Rate (Recall)", fontsize=20, fontweight='bold', labelpad=12)
plt.title("ROC Curve (One-Class SVM)", fontsize=24, fontweight='bold', pad=15)

# Ticks
plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

# ===== LEGEND (BOLD, NO BOX) =====
legend_font = FontProperties(weight='bold', size=16)

plt.legend(
    prop=legend_font,
    loc="lower right",
    frameon=False,
    labelspacing=0.8,
    handlelength=3,
    handletextpad=1
)

# Grid
plt.grid(True, linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()
# ===============================
# PRECISION–RECALL CURVE — One-Class SVM
# ===============================

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from matplotlib.font_manager import FontProperties

# Compute Precision–Recall curve
precision, recall, _ = precision_recall_curve(y_test_win, scores)

plt.figure(figsize=(8, 6), dpi=600)

plt.plot(
    recall, precision,
    linewidth=3,
    label=f"One-Class SVM (PR-AUC = {pr_auc:.3f})"
)

# Labels & title
plt.xlabel("Recall", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("Precision", fontsize=20, fontweight='bold', labelpad=12)
plt.title("Precision–Recall Curve (One-Class SVM)",
          fontsize=24, fontweight='bold', pad=15)

# Ticks
plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

# ===== LEGEND (BOLD, NO BOX) =====
legend_font = FontProperties(weight='bold', size=16)

plt.legend(
    prop=legend_font,
    loc="lower left",
    frameon=False,
    labelspacing=0.8,
    handlelength=3,
    handletextpad=1
)

# Grid
plt.grid(True, linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()
# ===============================
# FAR–RECALL CURVE — One-Class SVM (FIXED)
# ===============================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
from sklearn.metrics import roc_curve

# Compute FAR (FPR) and Recall (TPR)
fpr, tpr, _ = roc_curve(y_test_win, scores)

plt.figure(figsize=(8, 6), dpi=600)

plt.plot(
    fpr, tpr,
    linewidth=3,
    label="One-Class SVM"
)

# ===============================
# FAR–RECALL SAFE ANNOTATION (FINAL)
# ===============================

for far in [0.01, 0.05, 0.10]:
    idx = np.argmin(np.abs(fpr - far))
    x, y = fpr[idx], tpr[idx]

    plt.scatter(x, y, s=80, zorder=5)

    label_text = (
        f"FAR : {int(far*100)}%\n"
        f"Recall : {y:.2f}"
    )

    # Strong push away from origin if near axes
    if x < 0.05 and y < 0.1:
        offset = (45, 45)   # move up-right strongly
    else:
        offset = (35, -35)

    plt.annotate(
    label_text,
    xy=(x, y),
    xytext=offset,
    textcoords="offset points",
    fontsize=12,
    fontweight='bold',
    fontfamily='monospace',
    ha='left',
    va='bottom',
    clip_on=False,
    bbox=dict(
        boxstyle="round,pad=0.3",
        fc="white",
        ec="black",
        lw=0.8
    ),
    arrowprops=dict(
        arrowstyle='->',
        linewidth=1
    )
)

# Labels & title
plt.xlabel("False Alarm Rate (FAR)", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("Recall", fontsize=20, fontweight='bold', labelpad=12)
plt.title("Recall vs False Alarm Rate (One-Class SVM)",
          fontsize=22, fontweight='bold', pad=15)

# Ticks
plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

# Legend (bold, no box)
legend_font = FontProperties(weight='bold', size=16)
plt.legend(prop=legend_font, frameon=False, loc="lower right")

# Grid
plt.grid(True, linestyle='--', linewidth=1)

# Axis limits (important for spacing)
plt.xlim(-0.01, 1.02)
plt.ylim(-0.02, 1.05)

plt.tight_layout()
plt.show()
# ===============================
# ANOMALY SCORE DISTRIBUTION — One-Class SVM
# ===============================

import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties

# Ensure anomaly scores exist
anomaly_scores = -ocsvm.decision_function(X_test_flat)

# Separate scores by class
scores_normal  = anomaly_scores[y_test_win == 0]
scores_anomaly = anomaly_scores[y_test_win == 1]

plt.figure(figsize=(8, 6), dpi=600)

plt.hist(
    scores_normal,
    bins=15,
    alpha=0.7,
    density=True,
    label="Normal Windows",
    edgecolor="black"
)

plt.hist(
    scores_anomaly,
    bins=15,
    alpha=0.7,
    density=True,
    label="Eye-Closure Windows",
    edgecolor="black"
)

plt.xlabel("Anomaly Score", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("Density", fontsize=20, fontweight='bold', labelpad=12)
plt.title("Anomaly Score Distribution (One-Class SVM)",
          fontsize=22, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(prop=legend_font, frameon=False, loc="upper right")

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()


In [ ]:
# =====================================================
# Isolation Forest for EEG Eye State
# STRICT temporal split + ANY-based window labeling
# METRICS ONLY
# =====================================================

# -----------------------------
# 1. FIX RANDOMNESS
# -----------------------------
import numpy as np
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# -----------------------------
# 2. IMPORTS
# -----------------------------
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score,
    cohen_kappa_score, matthews_corrcoef
)

# -----------------------------
# 3. LOAD DATA (UPLOAD)
# -----------------------------
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
data = pd.read_csv(file_name)

X_raw = data.iloc[:, :-1].values
y_raw = data.iloc[:, -1].values

print("Raw data shape:", X_raw.shape)
print("Overall labels:", np.unique(y_raw, return_counts=True))

# -----------------------------
# 4. SCALE BEFORE WINDOWING
# -----------------------------
scaler = StandardScaler()
X_raw = scaler.fit_transform(X_raw)

# -----------------------------
# 5. STRICT TEMPORAL SPLIT
# -----------------------------
split_idx = int(0.8 * len(X_raw))

X_train_raw = X_raw[:split_idx]
y_train_raw = y_raw[:split_idx]

X_test_raw  = X_raw[split_idx:]
y_test_raw  = y_raw[split_idx:]

print("Train labels:", np.unique(y_train_raw, return_counts=True))
print("Test labels :", np.unique(y_test_raw, return_counts=True))

# -----------------------------
# 6. WINDOWING FUNCTION (ANY-based)
# -----------------------------
WINDOW_SIZE = 256
TRAIN_STRIDE = 16
TEST_STRIDE  = 256

def create_windows_any(X, y, window, stride):
    Xw, yw = [], []
    for i in range(0, len(X) - window + 1, stride):
        Xw.append(X[i:i + window])
        yw.append(int(np.any(y[i:i + window])))
    return np.array(Xw), np.array(yw)

# -----------------------------
# 7. WINDOWING
# -----------------------------
X_train_win, y_train_win = create_windows_any(
    X_train_raw, y_train_raw, WINDOW_SIZE, TRAIN_STRIDE
)

X_test_win, y_test_win = create_windows_any(
    X_test_raw, y_test_raw, WINDOW_SIZE, TEST_STRIDE
)

print("Train windows:", X_train_win.shape)
print("Test windows :", X_test_win.shape)
print("Windowed test labels:", np.unique(y_test_win, return_counts=True))

# -----------------------------
# 8. FLATTEN WINDOWS
# -----------------------------
X_train_flat = X_train_win.reshape(len(X_train_win), -1)
X_test_flat  = X_test_win.reshape(len(X_test_win), -1)

# =====================================================
# Isolation Forest — FULL ANOMALY METRICS
# STRICT temporal split + ANY-based window labeling
# =====================================================

import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve
)

# -----------------------------
# 1. TRAIN ISOLATION FOREST (NORMAL ONLY)
# -----------------------------
NORMAL_LABEL = 0

X_train_normal = X_train_flat[y_train_win == NORMAL_LABEL]

iso_forest = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    max_samples="auto",
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_normal)

# -----------------------------
# 2. ANOMALY SCORES
# -----------------------------
scores = -iso_forest.decision_function(X_test_flat)

# -----------------------------
# 3. PRIMARY METRICS
# -----------------------------
roc_auc = roc_auc_score(y_test_win, scores)
pr_auc  = average_precision_score(y_test_win, scores)

# -----------------------------
# 4. FAR–RECALL CURVE
# -----------------------------
fpr, tpr, thresholds = roc_curve(y_test_win, scores)

def recall_at_far(target_far):
    idx = np.where(fpr <= target_far)[0]
    return tpr[idx[-1]] if len(idx) > 0 else 0.0

recall_far_1  = recall_at_far(0.01)
recall_far_5  = recall_at_far(0.05)
recall_far_10 = recall_at_far(0.10)

# -----------------------------
# 5. FAR @ RECALL = 80%
# -----------------------------
def far_at_recall(target_recall):
    idx = np.where(tpr >= target_recall)[0]
    return fpr[idx[0]] if len(idx) > 0 else 1.0

far_recall_80 = far_at_recall(0.80)

# -----------------------------
# 6. EVENT DETECTION RATE (EDR)
# -----------------------------
detected_events = np.sum(scores[y_test_win == 1] > np.percentile(scores, 95))
total_events = np.sum(y_test_win == 1)
edr = detected_events / total_events if total_events > 0 else 0.0

# -----------------------------
# 7. FALSE ALARMS PER MINUTE
# -----------------------------
# EEG Eye State sampling rate ≈ 128 Hz
# Window = 256 → 2 seconds per window
SECONDS_PER_WINDOW = 2

false_alarms = np.sum(
    (scores > np.percentile(scores, 95)) & (y_test_win == 0)
)

false_alarms_per_min = false_alarms / (len(y_test_win) * SECONDS_PER_WINDOW / 60)

# -----------------------------
# 8. PRINT RESULTS
# -----------------------------
print("\n===== ISOLATION FOREST (FULL ANOMALY METRICS) =====")
print(f"ROC-AUC              : {roc_auc:.4f}")
print(f"PR-AUC               : {pr_auc:.4f}")
print(f"Recall @ FAR = 1%    : {recall_far_1:.4f}")
print(f"Recall @ FAR = 5%    : {recall_far_5:.4f}")
print(f"Recall @ FAR = 10%   : {recall_far_10:.4f}")
print(f"FAR @ Recall = 80%   : {far_recall_80:.4f}")
print(f"Event Detection Rate : {edr:.2f}")
print(f"False Alarms / Minute: {false_alarms_per_min:.2f}")

# ===============================
# ROC CURVE — Isolation Forest
# ===============================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from matplotlib.font_manager import FontProperties

# Compute ROC curve
fpr, tpr, _ = roc_curve(y_test_win, scores)

plt.figure(figsize=(8, 6), dpi=600)

plt.plot(
    fpr, tpr,
    linewidth=3,
    label=f"Isolation Forest (AUC = {roc_auc:.3f})"
)

plt.plot([0, 1], [0, 1], 'k--', linewidth=3)

plt.xlabel("False Positive Rate", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("True Positive Rate (Recall)", fontsize=20, fontweight='bold', labelpad=12)
plt.title("ROC Curve (Isolation Forest)",
          fontsize=24, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(
    prop=legend_font,
    frameon=False,
    loc="lower right"
)

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

# ===============================
# ROC CURVE — Isolation Forest
# ===============================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from matplotlib.font_manager import FontProperties

# Compute ROC curve
fpr, tpr, _ = roc_curve(y_test_win, scores)

plt.figure(figsize=(8, 6), dpi=600)

plt.plot(
    fpr, tpr,
    linewidth=3,
    label=f"Isolation Forest (AUC = {roc_auc:.3f})"
)

plt.plot([0, 1], [0, 1], 'k--', linewidth=3)

plt.xlabel("False Positive Rate", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("True Positive Rate (Recall)", fontsize=20, fontweight='bold', labelpad=12)
plt.title("ROC Curve (Isolation Forest)",
          fontsize=24, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(
    prop=legend_font,
    frameon=False,
    loc="lower right"
)

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()


# ===============================
# ROC CURVE — Isolation Forest
# ===============================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from matplotlib.font_manager import FontProperties

# Compute ROC curve
fpr, tpr, _ = roc_curve(y_test_win, scores)

plt.figure(figsize=(8, 6), dpi=600)

plt.plot(
    fpr, tpr,
    linewidth=3,
    label=f"Isolation Forest (AUC = {roc_auc:.3f})"
)

plt.plot([0, 1], [0, 1], 'k--', linewidth=3)

plt.xlabel("False Positive Rate", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("True Positive Rate (Recall)", fontsize=20, fontweight='bold', labelpad=12)
plt.title("ROC Curve (Isolation Forest)",
          fontsize=24, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(
    prop=legend_font,
    frameon=False,
    loc="lower right"
)

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()


# ===============================
# ANOMALY SCORE DISTRIBUTION — Isolation Forest
# ===============================

import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties

# Separate scores by class
scores_normal  = scores[y_test_win == 0]
scores_anomaly = scores[y_test_win == 1]

plt.figure(figsize=(8, 6), dpi=600)

# Normal windows
plt.hist(
    scores_normal,
    bins=15,
    density=True,
    alpha=0.7,
    label="Normal Windows",
    edgecolor="black"
)

# Anomalous windows
plt.hist(
    scores_anomaly,
    bins=15,
    density=True,
    alpha=0.7,
    label="Eye-Closure Windows",
    edgecolor="black"
)

# Labels & title
plt.xlabel("Anomaly Score", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("Density", fontsize=20, fontweight='bold', labelpad=12)
plt.title("Anomaly Score Distribution (Isolation Forest)",
          fontsize=22, fontweight='bold', pad=15)

# Ticks
plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

# Legend (bold, no box)
legend_font = FontProperties(weight='bold', size=16)
plt.legend(
    prop=legend_font,
    frameon=False,
    loc="upper right"
)

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()


In [ ]:
# =====================================================
# Autoencoder for EEG Eye State
# STRICT temporal split + ANY-based window labeling
# FULL PIPELINE | METRICS ONLY
# =====================================================

# -----------------------------
# 1. FIX RANDOMNESS
# -----------------------------
import numpy as np
import random
import tensorflow as tf

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------------
# 2. IMPORTS
# -----------------------------
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# -----------------------------
# 3. LOAD DATA (UPLOAD CSV)
# -----------------------------
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
data = pd.read_csv(file_name)

X_raw = data.iloc[:, :-1].values   # EEG channels
y_raw = data.iloc[:, -1].values   # eyeDetection label

print("Raw data shape:", X_raw.shape)
print("Overall labels:", np.unique(y_raw, return_counts=True))

# -----------------------------
# 4. SCALE BEFORE WINDOWING
# -----------------------------
scaler = StandardScaler()
X_raw = scaler.fit_transform(X_raw)

# -----------------------------
# 5. STRICT TEMPORAL SPLIT
# -----------------------------
split_idx = int(0.8 * len(X_raw))

X_train_raw = X_raw[:split_idx]
y_train_raw = y_raw[:split_idx]

X_test_raw  = X_raw[split_idx:]
y_test_raw  = y_raw[split_idx:]

print("Train labels:", np.unique(y_train_raw, return_counts=True))
print("Test labels :", np.unique(y_test_raw, return_counts=True))

# -----------------------------
# 6. WINDOWING FUNCTION (ANY-based)
# -----------------------------
WINDOW_SIZE = 256
TRAIN_STRIDE = 16
TEST_STRIDE  = 256

def create_windows_any(X, y, window, stride):
    Xw, yw = [], []
    for i in range(0, len(X) - window + 1, stride):
        Xw.append(X[i:i + window])
        yw.append(int(np.any(y[i:i + window])))
    return np.array(Xw), np.array(yw)

# -----------------------------
# 7. WINDOWING
# -----------------------------
X_train_win, y_train_win = create_windows_any(
    X_train_raw, y_train_raw, WINDOW_SIZE, TRAIN_STRIDE
)

X_test_win, y_test_win = create_windows_any(
    X_test_raw, y_test_raw, WINDOW_SIZE, TEST_STRIDE
)

print("Train windows:", X_train_win.shape)
print("Test windows :", X_test_win.shape)
print("Windowed test labels:", np.unique(y_test_win, return_counts=True))

# -----------------------------
# 8. FLATTEN WINDOWS
# -----------------------------
X_train_flat = X_train_win.reshape(len(X_train_win), -1)
X_test_flat  = X_test_win.reshape(len(X_test_win), -1)

# -----------------------------
# 9. TRAIN AUTOENCODER (NORMAL ONLY)
# -----------------------------
NORMAL_LABEL = 0
X_train_normal = X_train_flat[y_train_win == NORMAL_LABEL]

print("Training on normal windows:", X_train_normal.shape)

input_dim = X_train_normal.shape[1]

inputs = Input(shape=(input_dim,))
x = Dense(128, activation='relu')(inputs)
x = Dense(64, activation='relu')(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(input_dim, activation='linear')(x)

autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer=Adam(1e-3), loss='mse')

autoencoder.summary()

# -----------------------------
# 10. TRAIN
# -----------------------------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

# -----------------------------
# 11. RECONSTRUCTION ERROR = ANOMALY SCORE
# -----------------------------
X_test_recon = autoencoder.predict(X_test_flat, verbose=0)
scores = np.mean((X_test_flat - X_test_recon) ** 2, axis=1)

# -----------------------------
# 12. PRIMARY METRICS
# -----------------------------
roc_auc = roc_auc_score(y_test_win, scores)
pr_auc  = average_precision_score(y_test_win, scores)

# -----------------------------
# 13. FAR–RECALL CURVE
# -----------------------------
fpr, tpr, thresholds = roc_curve(y_test_win, scores)

def recall_at_far(target_far):
    idx = np.where(fpr <= target_far)[0]
    return tpr[idx[-1]] if len(idx) > 0 else 0.0

recall_far_1  = recall_at_far(0.01)
recall_far_5  = recall_at_far(0.05)
recall_far_10 = recall_at_far(0.10)

# -----------------------------
# 14. FAR @ RECALL = 80%
# -----------------------------
def far_at_recall(target_recall):
    idx = np.where(tpr >= target_recall)[0]
    return fpr[idx[0]] if len(idx) > 0 else 1.0

far_recall_80 = far_at_recall(0.80)

# -----------------------------
# 15. EVENT DETECTION RATE (EDR)
# -----------------------------
threshold = np.percentile(scores, 95)
detected_events = np.sum((scores >= threshold) & (y_test_win == 1))
total_events = np.sum(y_test_win == 1)
edr = detected_events / total_events if total_events > 0 else 0.0

# -----------------------------
# 16. FALSE ALARMS / MINUTE
# -----------------------------
SECONDS_PER_WINDOW = 2   # 256 samples @ 128 Hz
false_alarms = np.sum((scores >= threshold) & (y_test_win == 0))
false_alarms_per_min = false_alarms / (len(y_test_win) * SECONDS_PER_WINDOW / 60)

# -----------------------------
# 17. PRINT RESULTS
# -----------------------------
print("\n===== AUTOENCODER (FULL ANOMALY METRICS) =====")
print(f"ROC-AUC              : {roc_auc:.4f}")
print(f"PR-AUC               : {pr_auc:.4f}")
print(f"Recall @ FAR = 1%    : {recall_far_1:.4f}")
print(f"Recall @ FAR = 5%    : {recall_far_5:.4f}")
print(f"Recall @ FAR = 10%   : {recall_far_10:.4f}")
print(f"FAR @ Recall = 80%   : {far_recall_80:.4f}")
print(f"Event Detection Rate : {edr:.2f}")
print(f"False Alarms / Minute: {false_alarms_per_min:.2f}")


# ===============================
# ROC CURVE — Autoencoder
# ===============================
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from matplotlib.font_manager import FontProperties

fpr, tpr, _ = roc_curve(y_test_win, scores)

plt.figure(figsize=(8, 6), dpi=600)
plt.plot(fpr, tpr, linewidth=3, label=f"Autoencoder (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], 'k--', linewidth=3)

plt.xlabel("False Positive Rate", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("True Positive Rate (Recall)", fontsize=20, fontweight='bold', labelpad=12)
plt.title("ROC Curve (Autoencoder)", fontsize=24, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(prop=legend_font, frameon=False, loc="lower right")

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()


# ===============================
# PRECISION–RECALL CURVE — Autoencoder
# ===============================
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_test_win, scores)

plt.figure(figsize=(8, 6), dpi=600)
plt.plot(recall, precision, linewidth=3,
         label=f"Autoencoder (PR-AUC = {pr_auc:.3f})")

plt.xlabel("Recall", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("Precision", fontsize=20, fontweight='bold', labelpad=12)
plt.title("Precision–Recall Curve (Autoencoder)",
          fontsize=24, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(prop=legend_font, frameon=False, loc="lower left")

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()


# ===============================
# FAR–RECALL CURVE — Autoencoder
# ===============================
import numpy as np

plt.figure(figsize=(8, 6), dpi=600)
plt.plot(fpr, tpr, linewidth=3, label="Autoencoder")

far_points = [0.01, 0.05, 0.10]

for far in far_points:
    idx = np.argmin(np.abs(fpr - far))
    x, y = fpr[idx], tpr[idx]

    plt.scatter(x, y, s=80, zorder=5)

    plt.annotate(
        f"FAR: {int(far*100)}%\nRecall: {y:.2f}",
        xy=(x, y),
        xytext=(35, -35),
        textcoords="offset points",
        fontsize=12,
        fontweight='bold',
        fontfamily='monospace',
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=0.8),
        arrowprops=dict(arrowstyle="->", linewidth=1)
    )

plt.xlabel("False Alarm Rate (FAR)", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("Recall", fontsize=20, fontweight='bold', labelpad=12)
plt.title("Recall vs False Alarm Rate (Autoencoder)",
          fontsize=24, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(prop=legend_font, frameon=False, loc="lower right")

plt.grid(True, linestyle='--', linewidth=1)
plt.xlim(-0.01, 1.02)
plt.ylim(-0.02, 1.05)

plt.tight_layout()
plt.show()


# ===============================
# ANOMALY SCORE DISTRIBUTION — Autoencoder
# ===============================

scores_normal  = scores[y_test_win == 0]
scores_anomaly = scores[y_test_win == 1]

plt.figure(figsize=(8, 6), dpi=600)

plt.hist(scores_normal, bins=15, density=True, alpha=0.7,
         label="Normal Windows", edgecolor="black")

plt.hist(scores_anomaly, bins=15, density=True, alpha=0.7,
         label="Eye-Closure Windows", edgecolor="black")

plt.xlabel("Reconstruction Error", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("Density", fontsize=20, fontweight='bold', labelpad=12)
plt.title("Anomaly Score Distribution (Autoencoder)",
          fontsize=24, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(prop=legend_font, frameon=False, loc="upper right")

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()


In [ ]:
# =====================================================
# PROPOSED MODEL: Temporal–Spatial Autoencoder (TSAE)
# EEG Eye State Anomaly Detection
# STRICT | LEAKAGE-FREE | METRICS ONLY
# =====================================================

# -----------------------------
# 1. FIX RANDOMNESS
# -----------------------------
import numpy as np
import random
import tensorflow as tf

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------------
# 2. IMPORTS
# -----------------------------
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, LSTM, Bidirectional,
    RepeatVector, TimeDistributed
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# -----------------------------
# 3. LOAD DATA (UPLOAD CSV)
# -----------------------------
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
data = pd.read_csv(file_name)

X_raw = data.iloc[:, :-1].values   # EEG channels
y_raw = data.iloc[:, -1].values   # eyeDetection

print("Raw EEG shape:", X_raw.shape)
print("Overall labels:", np.unique(y_raw, return_counts=True))

# -----------------------------
# 4. SCALE BEFORE WINDOWING
# -----------------------------
scaler = StandardScaler()
X_raw = scaler.fit_transform(X_raw)

# -----------------------------
# 5. STRICT TEMPORAL SPLIT
# -----------------------------
split_idx = int(0.8 * len(X_raw))

X_train_raw = X_raw[:split_idx]
y_train_raw = y_raw[:split_idx]

X_test_raw  = X_raw[split_idx:]
y_test_raw  = y_raw[split_idx:]

print("Train labels:", np.unique(y_train_raw, return_counts=True))
print("Test labels :", np.unique(y_test_raw, return_counts=True))

# -----------------------------
# 6. WINDOWING (ANY-based)
# -----------------------------
WINDOW_SIZE = 256
TRAIN_STRIDE = 16
TEST_STRIDE  = 256

def create_windows_any(X, y, window, stride):
    Xw, yw = [], []
    for i in range(0, len(X) - window + 1, stride):
        Xw.append(X[i:i + window])
        yw.append(int(np.any(y[i:i + window])))
    return np.array(Xw), np.array(yw)

X_train_win, y_train_win = create_windows_any(
    X_train_raw, y_train_raw, WINDOW_SIZE, TRAIN_STRIDE
)

X_test_win, y_test_win = create_windows_any(
    X_test_raw, y_test_raw, WINDOW_SIZE, TEST_STRIDE
)

print("Train windows:", X_train_win.shape)
print("Test windows :", X_test_win.shape)
print("Test window labels:", np.unique(y_test_win, return_counts=True))

# -----------------------------
# 7. TRAIN ON NORMAL WINDOWS ONLY
# -----------------------------
NORMAL_LABEL = 0
X_train_normal = X_train_win[y_train_win == NORMAL_LABEL]

print("Training windows (normal only):", X_train_normal.shape)

# -----------------------------
# 8. PROPOSED MODEL ARCHITECTURE
# -----------------------------
timesteps = X_train_normal.shape[1]
channels  = X_train_normal.shape[2]

inputs = Input(shape=(timesteps, channels))

# 🔹 Temporal Encoder
x = Bidirectional(LSTM(64, return_sequences=False))(inputs)
latent = Dense(32, activation='relu')(x)

# 🔹 Temporal–Spatial Decoder
x = RepeatVector(timesteps)(latent)
x = LSTM(64, return_sequences=True)(x)
outputs = TimeDistributed(Dense(channels))(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=Adam(1e-3),
    loss='mse'
)

model.summary()

# -----------------------------
# 9. TRAIN MODEL
# -----------------------------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

model.fit(
    X_train_normal, X_train_normal,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

# -----------------------------
# 10. RECONSTRUCTION ERROR
# -----------------------------
X_test_recon = model.predict(X_test_win, verbose=0)
scores = np.mean((X_test_win - X_test_recon) ** 2, axis=(1, 2))

# -----------------------------
# 11. PRIMARY METRICS
# -----------------------------
roc_auc = roc_auc_score(y_test_win, scores)
pr_auc  = average_precision_score(y_test_win, scores)

# -----------------------------
# 12. FAR–RECALL ANALYSIS
# -----------------------------
fpr, tpr, thresholds = roc_curve(y_test_win, scores)

def recall_at_far(far):
    idx = np.where(fpr <= far)[0]
    return tpr[idx[-1]] if len(idx) > 0 else 0.0

recall_far_1  = recall_at_far(0.01)
recall_far_5  = recall_at_far(0.05)
recall_far_10 = recall_at_far(0.10)

def far_at_recall(rec):
    idx = np.where(tpr >= rec)[0]
    return fpr[idx[0]] if len(idx) > 0 else 1.0

far_recall_80 = far_at_recall(0.80)

# -----------------------------
# 13. EVENT DETECTION RATE
# -----------------------------
threshold = np.percentile(scores, 95)
detected = np.sum((scores >= threshold) & (y_test_win == 1))
total_events = np.sum(y_test_win == 1)
edr = detected / total_events if total_events > 0 else 0.0

# -----------------------------
# 14. FALSE ALARMS / MINUTE
# -----------------------------
SECONDS_PER_WINDOW = 2
false_alarms = np.sum((scores >= threshold) & (y_test_win == 0))
false_alarms_per_min = false_alarms / (len(y_test_win) * SECONDS_PER_WINDOW / 60)

# -----------------------------
# 15. PRINT RESULTS
# -----------------------------
print("\n===== PROPOSED TSAE (FULL ANOMALY METRICS) =====")
print(f"ROC-AUC              : {roc_auc:.4f}")
print(f"PR-AUC               : {pr_auc:.4f}")
print(f"Recall @ FAR = 1%    : {recall_far_1:.4f}")
print(f"Recall @ FAR = 5%    : {recall_far_5:.4f}")
print(f"Recall @ FAR = 10%   : {recall_far_10:.4f}")
print(f"FAR @ Recall = 80%   : {far_recall_80:.4f}")
print(f"Event Detection Rate : {edr:.2f}")
print(f"False Alarms / Minute: {false_alarms_per_min:.2f}")


# =====================================================
# 16. AVG INFERENCE TIME / WINDOW (ms)
# =====================================================
import time

def measure_avg_inference_time(model, X_windows, runs=100):
    """
    Measures average inference time per window (ms)
    """
    times = []

    # Warm-up (important for TensorFlow)
    _ = model.predict(X_windows[:1], verbose=0)

    for i in range(min(runs, len(X_windows))):
        x = X_windows[i:i+1]   # shape: (1, 256, channels)

        start = time.perf_counter()
        _ = model.predict(x, verbose=0)
        end = time.perf_counter()

        times.append(end - start)

    return np.mean(times) * 1000  # ms


avg_inference_time_ms = measure_avg_inference_time(
    model,

    X_test_win,
    runs=100
)

print(f"Avg. Inference Time / Window (ms): {avg_inference_time_ms:.3f}")


import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from matplotlib.font_manager import FontProperties

# ROC data
fpr, tpr, _ = roc_curve(y_test_win, scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6), dpi=600)

plt.plot(
    fpr, tpr,
    linewidth=3,
    label=f"TSAE (AUC = {roc_auc:.3f})"
)

plt.plot([0, 1], [0, 1], 'k--', linewidth=3)

plt.xlabel("False Positive Rate", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("True Positive Rate", fontsize=20, fontweight='bold', labelpad=12)
plt.title("ROC Curve (Proposed TSAE)", fontsize=24, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(prop=legend_font, frameon=False, loc="lower right")

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()


import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
from matplotlib.font_manager import FontProperties

# Precision–Recall data
precision, recall, _ = precision_recall_curve(y_test_win, scores)
pr_auc = average_precision_score(y_test_win, scores)

plt.figure(figsize=(8, 6), dpi=600)

plt.plot(
    recall, precision,
    linewidth=3,
    label=f"TSAE (AP = {pr_auc:.3f})"
)

plt.xlabel("Recall", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("Precision", fontsize=20, fontweight='bold', labelpad=12)
plt.title("Precision–Recall Curve (Proposed TSAE)",
          fontsize=22, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=16)
plt.legend(prop=legend_font, frameon=False, loc="lower left")

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from matplotlib.font_manager import FontProperties

# Compute ROC
fpr, tpr, _ = roc_curve(y_test_win, scores)

far = fpr * 100
recall = tpr

plt.figure(figsize=(8, 6), dpi=600)

# FAR–Recall curve
plt.plot(far, recall, linewidth=3, label="Autoencoder")

# ---- Select FAR = 10% operating point ----
target_far = 10
idx = np.argmin(np.abs(far - target_far))

# Marker
plt.scatter(far[idx], recall[idx], s=100, color="green", zorder=6)

# ---- Box + Arrow annotation ----
plt.annotate(
    f"FAR : {target_far}%\nRecall : {recall[idx]:.2f}",
    xy=(far[idx], recall[idx]),              # actual point
    xytext=(far[idx] + 8, recall[idx] + 0.15),  # text position
    textcoords='data',
    fontsize=16,
    fontweight='bold',
    bbox=dict(
        boxstyle="round,pad=0.35",
        fc="white",
        ec="black",
        linewidth=1
    ),
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.5,
        color="black"
    )
)

# Labels & title
plt.xlabel("False Alarm Rate (FAR)", fontsize=20, fontweight='bold', labelpad=12)
plt.ylabel("Recall", fontsize=20, fontweight='bold', labelpad=12)
plt.title("Recall vs False Alarm Rate (TSAE)",
          fontsize=24, fontweight='bold', pad=15)

# Ticks
plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

# Legend
legend_font = FontProperties(weight='bold', size=18)
plt.legend(prop=legend_font, frameon=False, loc="lower right")

# Grid
plt.grid(True, linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()


# FAR–Recall Curve — Proposed TSAE

plt.figure(figsize=(8, 6), dpi=100)

plt.plot(far, recall, linewidth=3, label="Proposed TSAE")

# ---- Select FAR = 10% operating point ----
target_far = 10
idx = np.argmin(np.abs(far - target_far))

plt.scatter(
    far[idx],
    recall[idx],
    s=100,
    color="red",
    zorder=6
)

plt.annotate(
    f"FAR : {target_far}%\nRecall : {recall[idx]:.2f}",
    xy=(far[idx], recall[idx]),
    xytext=(far[idx] + 8, recall[idx] + 0.15),
    textcoords='data',
    fontsize=16,
    fontweight='bold',
    bbox=dict(
        boxstyle="round,pad=0.35",
        fc="white",
        ec="black",
        linewidth=1
    ),
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.5
    )
)

plt.xlabel("False Alarm Rate (FAR)", fontsize=22, fontweight='bold', labelpad=12)
plt.ylabel("Recall", fontsize=24, fontweight='bold', labelpad=12)
plt.title("Recall vs False Alarm Rate (Proposed TSAE)",
          fontsize=22, fontweight='bold', pad=15)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

legend_font = FontProperties(weight='bold', size=20)
plt.legend(prop=legend_font, frameon=False, loc="lower right")

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()


import matplotlib.pyplot as plt
import numpy as np
from matplotlib.font_manager import FontProperties

# Separate anomaly scores by class
scores_normal  = scores[y_test_win == 0]
scores_anomaly = scores[y_test_win == 1]

plt.figure(figsize=(8, 6), dpi=600)

# Normal windows
plt.hist(
    scores_normal,
    bins=15,
    density=True,
    alpha=0.75,
    label="Normal Windows",
    edgecolor="black"
)

# Eye-closure (anomalous) windows
plt.hist(
    scores_anomaly,
    bins=15,
    density=True,
    alpha=0.75,
    label="Eye-Closure Windows",
    edgecolor="black"
)

plt.xlabel("Anomaly Score (Reconstruction Error)",
           fontsize=20, fontweight='bold', labelpad=5)
plt.ylabel("Density",
           fontsize=20, fontweight='bold', labelpad=5)

plt.title("Anomaly Score Distribution (Proposed TSAE)",
          fontsize=22, fontweight='bold', pad=10)

plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

# ---- Spine control (prevents thick black bar artifacts) ----
ax = plt.gca()
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

legend_font = FontProperties(weight='bold', size=16)
plt.legend(prop=legend_font, frameon=False, loc="upper right")

# ---- Spine control (ALL SIDES VISIBLE) ----
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.5)

plt.grid(True, linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()




In [ ]:
# ===============================
# STEP 1: BASIC SETUP (SAFE)
# ===============================

import numpy as np
import random
import tensorflow as tf

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Runtime connected successfully.")


# ===============================
# STEP 2: UPLOAD & LOAD DATASET
# ===============================

import pandas as pd
import numpy as np
from google.colab import files

# Upload CSV
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Loaded file:", file_name)

# Read CSV
data = pd.read_csv(file_name)

# Split features & labels
X_raw = data.iloc[:, :-1].values
y_raw = data.iloc[:, -1].values

print("Data loaded successfully.")
print("X shape:", X_raw.shape)
print("y shape:", y_raw.shape)
print("Label distribution:", np.unique(y_raw, return_counts=True))


# ===============================
# STEP 3: SCALE + STRICT TEMPORAL SPLIT
# ===============================

from sklearn.preprocessing import StandardScaler

# Scale BEFORE windowing (as in your original code)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Strict temporal split (80%)
split_idx = int(0.8 * len(X_scaled))

X_train_raw = X_scaled[:split_idx]
y_train_raw = y_raw[:split_idx]

X_test_raw  = X_scaled[split_idx:]
y_test_raw  = y_raw[split_idx:]

print("Scaling + temporal split done.")
print("Train raw shape:", X_train_raw.shape)
print("Test raw shape :", X_test_raw.shape)
print("Train labels:", np.unique(y_train_raw, return_counts=True))
print("Test labels :", np.unique(y_test_raw, return_counts=True))


# ===============================
# STEP 4: WINDOWING (ANY-based)
# ===============================

import numpy as np

WINDOW_SIZE = 256
TRAIN_STRIDE = 16
TEST_STRIDE  = 256

def create_windows_any(X, y, window, stride):
    Xw, yw = [], []
    for i in range(0, len(X) - window + 1, stride):
        Xw.append(X[i:i + window])
        yw.append(int(np.any(y[i:i + window])))
    return np.array(Xw), np.array(yw)

# Train windows
X_train_win, y_train_win = create_windows_any(
    X_train_raw, y_train_raw, WINDOW_SIZE, TRAIN_STRIDE
)

# Test windows
X_test_win, y_test_win = create_windows_any(
    X_test_raw, y_test_raw, WINDOW_SIZE, TEST_STRIDE
)

print("Windowing done successfully.")
print("Train windows shape:", X_train_win.shape)
print("Test windows shape :", X_test_win.shape)
print("Test window labels:", np.unique(y_test_win, return_counts=True))

# =====================================================
# STEP 5: TRAIN MODELS & COLLECT ANOMALY SCORES ONLY
# =====================================================

from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_curve
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, RepeatVector, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import numpy as np

# ---------- Flatten for classical models ----------
X_train_flat = X_train_win.reshape(len(X_train_win), -1)
X_test_flat  = X_test_win.reshape(len(X_test_win), -1)

# =====================================================
# 1️⃣ ONE-CLASS SVM
# =====================================================
X_train_norm = X_train_flat[y_train_win == 0]

ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.05)
ocsvm.fit(X_train_norm)

scores_ocsvm = -ocsvm.decision_function(X_test_flat)

print("One-Class SVM done.")

# =====================================================
# 2️⃣ ISOLATION FOREST
# =====================================================
iso = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)
iso.fit(X_train_norm)

scores_iforest = -iso.decision_function(X_test_flat)

print("Isolation Forest done.")

# =====================================================
# 3️⃣ AUTOENCODER (MLP)
# =====================================================
input_dim = X_train_norm.shape[1]

inp = Input(shape=(input_dim,))
x = Dense(128, activation='relu')(inp)
x = Dense(64, activation='relu')(x)
x = Dense(128, activation='relu')(x)
out = Dense(input_dim)(x)

ae = Model(inp, out)
ae.compile(optimizer=Adam(1e-3), loss='mse')

ae.fit(
    X_train_norm, X_train_norm,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    shuffle=True,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

X_recon = ae.predict(X_test_flat, verbose=0)
scores_ae = np.mean((X_test_flat - X_recon) ** 2, axis=1)

print("Autoencoder done.")

# =====================================================
# 4️⃣ TSAE (PROPOSED)
# =====================================================
timesteps = X_train_win.shape[1]
channels  = X_train_win.shape[2]

X_train_tsae = X_train_win[y_train_win == 0]

inp = Input(shape=(timesteps, channels))
x = Bidirectional(LSTM(64, return_sequences=False))(inp)
z = Dense(32, activation='relu')(x)
x = RepeatVector(timesteps)(z)
x = LSTM(64, return_sequences=True)(x)
out = TimeDistributed(Dense(channels))(x)

tsae = Model(inp, out)
tsae.compile(optimizer=Adam(1e-3), loss='mse')

tsae.fit(
    X_train_tsae, X_train_tsae,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    shuffle=True,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

X_tsae_recon = tsae.predict(X_test_win, verbose=0)
scores_tsae = np.mean((X_test_win - X_tsae_recon) ** 2, axis=(1,2))

print("TSAE done.")

# =====================================================
# STEP 6: COMBINED ROC CURVES (FINAL FIGURE)
# =====================================================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from matplotlib.font_manager import FontProperties

# Compute ROC curves
fpr_svm, tpr_svm, _ = roc_curve(y_test_win, scores_ocsvm)
fpr_if,  tpr_if,  _ = roc_curve(y_test_win, scores_iforest)
fpr_ae,  tpr_ae,  _ = roc_curve(y_test_win, scores_ae)
fpr_ts,  tpr_ts,  _ = roc_curve(y_test_win, scores_tsae)

# AUC values
auc_svm = auc(fpr_svm, tpr_svm)
auc_if  = auc(fpr_if,  tpr_if)
auc_ae  = auc(fpr_ae,  tpr_ae)
auc_ts  = auc(fpr_ts,  tpr_ts)

# Plot
plt.figure(figsize=(9, 7), dpi=100)

plt.plot(fpr_svm, tpr_svm, linewidth=2.5,
         label=f"One-Class SVM (AUC = {auc_svm:.3f})")

plt.plot(fpr_if, tpr_if, linewidth=2.5,
         label=f"Isolation Forest (AUC = {auc_if:.3f})")

plt.plot(fpr_ae, tpr_ae, linewidth=2.5,
         label=f"Autoencoder (AUC = {auc_ae:.3f})")

# Proposed model highlighted
plt.plot(fpr_ts, tpr_ts, linewidth=4,
         label=f"Proposed TSAE (AUC = {auc_ts:.3f})")

# Diagonal
plt.plot([0, 1], [0, 1], 'k--', linewidth=3)

# Labels and title
plt.xlabel("False Positive Rate", fontsize=24, fontweight='bold', labelpad=12)
plt.ylabel("True Positive Rate", fontsize=24, fontweight='bold', labelpad=12)
plt.title("ROC Curves Comparison",
          fontsize=28, fontweight='bold', pad=15)

# Ticks
plt.xticks(fontsize=18, fontweight='bold')
plt.yticks(fontsize=18, fontweight='bold')

# Legend
legend_font = FontProperties(weight='bold', size=18)
plt.legend(prop=legend_font, frameon=False, loc="lower right")

# Grid
plt.grid(True, linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()


import numpy as np

np.save("y_test_win.npy", y_test_win)
np.save("scores_ocsvm.npy", scores_ocsvm)
np.save("scores_iforest.npy", scores_iforest)
np.save("scores_ae.npy", scores_ae)
np.save("scores_tsae.npy", scores_tsae)


# =====================================================
# STEP 7: COMBINED PRECISION–RECALL CURVES (FINAL)
# =====================================================

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
from matplotlib.font_manager import FontProperties

# Compute PR curves
prec_svm, rec_svm, _ = precision_recall_curve(y_test_win, scores_ocsvm)
prec_if,  rec_if,  _ = precision_recall_curve(y_test_win, scores_iforest)
prec_ae,  rec_ae,  _ = precision_recall_curve(y_test_win, scores_ae)
prec_ts,  rec_ts,  _ = precision_recall_curve(y_test_win, scores_tsae)

# Average Precision (AP / PR-AUC)
ap_svm = average_precision_score(y_test_win, scores_ocsvm)
ap_if  = average_precision_score(y_test_win, scores_iforest)
ap_ae  = average_precision_score(y_test_win, scores_ae)
ap_ts  = average_precision_score(y_test_win, scores_tsae)

# Plot
plt.figure(figsize=(9, 7), dpi=100)

plt.plot(rec_svm, prec_svm, linewidth=3,
         label=f"One-Class SVM (AP = {ap_svm:.3f})")

plt.plot(rec_if, prec_if, linewidth=3,
         label=f"Isolation Forest (AP = {ap_if:.3f})")

plt.plot(rec_ae, prec_ae, linewidth=3,
         label=f"Autoencoder (AP = {ap_ae:.3f})")

# Proposed model highlighted
plt.plot(rec_ts, prec_ts, linewidth=4,
         label=f"Proposed TSAE (AP = {ap_ts:.3f})")

# Labels & title
plt.xlabel("Recall", fontsize=28, fontweight='bold', labelpad=12)
plt.ylabel("Precision", fontsize=28, fontweight='bold', labelpad=12)
plt.title("Precision–Recall Curves Comparison",
          fontsize=26, fontweight='bold', pad=15)

# Ticks
plt.xticks(fontsize=22, fontweight='bold')
plt.yticks(fontsize=22, fontweight='bold')

# Legend
legend_font = FontProperties(weight='bold', size=17)
plt.legend(prop=legend_font, frameon=False, loc="lower left")

# Grid
plt.grid(True, linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()


np.save("ap_values.npy", np.array([ap_svm, ap_if, ap_ae, ap_ts]))

